In [2]:
import os
import logging
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

# Silenciar logs informativos de JVM/Spark para salidas limpias
os.environ["PYSPARK_SUBMIT_ARGS"] = "--conf spark.ui.showConsoleProgress=false pyspark-shell"
logging.getLogger("py4j").setLevel(logging.ERROR)

# Crear SparkSession con 2 hilos (1 receptor + 1 procesador, mínimo para streaming)
spark = (
    SparkSession.builder
    .appName("Streaming IABD WordCount")
    .master("local[2]")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark Streaming listo - Versión: {spark.version} | Master: {spark.sparkContext.master}")

Spark Streaming listo - Versión: 3.5.3 | Master: local[2]


In [ ]:
Caso 1 — Hola Spark Streaming (WordCount en tiempo real)

In [3]:
# Caso 1: Hola Spark Streaming (WordCount en tiempo real)
lineasDF = (
    spark.readStream
    .format("socket")
    .option("host", "localhost")
    .option("port", 9999)
    .load()
)

palabrasDF = lineasDF.select(explode(split(lineasDF.value, " ")).alias("palabra"))
cantidadDF = palabrasDF.groupBy("palabra").count()

wordCountQuery = (
    cantidadDF.writeStream
    .format("console")
    .outputMode("complete")
    .start()
)

wordCountQuery.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|streaming|    1|
|    spark|    1|
|     hola|    1|
+---------+-----+

-------------------------------------------
Batch: 2
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|streaming|    1|
|    spark|    2|
|     hola|    2|
|    mundo|    1|
+---------+-----+

-------------------------------------------
Batch: 3
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|   nombre|    1|
|streaming|    1|
|       es|    1|
|    spark|    2|
|     hola|    2|
|       mi|    1|
| valentin|    3|
|    mundo|    1|
+---------+-----+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
Comparar el modo de salida update

In [4]:
# Ejercicio 1: Comparación de modos de salida - Modo "update"
# Solo imprime las palabras nuevas o cuyo conteo cambió en el micro-batch actual
wordCountUpdate = (
    cantidadDF.writeStream
    .format("console")
    .outputMode("update")
    .start()
)

wordCountUpdate.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 4
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|   nombre|    1|
|streaming|    1|
|       es|    1|
|    spark|    3|
|     hola|    2|
|       mi|    1|
|    nuevo|    1|
| valentin|    3|
|    mundo|    1|
|     dato|    1|
+---------+-----+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
transformación sin estado filter + upper para usar outputMode("append")

In [5]:
# Ejercicio 1 (cont.): Modo de salida "append" (solo para transformaciones sin estado)
from pyspark.sql.functions import upper, length

palabrasLargasDF = (
    palabrasDF
    .filter(length("palabra") > 4)
    .select(upper("palabra").alias("palabra_mayuscula"))
)

appendQuery = (
    palabrasLargasDF.writeStream
    .format("console")
    .outputMode("append")
    .start()
)

appendQuery.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------------+
|palabra_mayuscula|
+-----------------+
+-----------------+

-------------------------------------------
Batch: 5
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|   nombre|    1|
|streaming|    2|
|       es|    1|
|   apache|    1|
|    spark|    4|
|     hola|    2|
|       mi|    1|
|    nuevo|    1|
|      sol|    1|
| valentin|    3|
|    mundo|    1|
|     dato|    1|
+---------+-----+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [6]:
# Detener cualquier stream previo activo en Spark
for q in spark.streams.active:
    q.stop()

# Ejecutar limpio el modo "append"
appendQuery = (
    palabrasLargasDF.writeStream
    .format("console")
    .outputMode("append")
    .start()
)

appendQuery.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----------------+
|palabra_mayuscula|
+-----------------+
+-----------------+

-------------------------------------------
Batch: 1
-------------------------------------------
+-----------------+
|palabra_mayuscula|
+-----------------+
|           APACHE|
|            SPARK|
|        STREAMING|
+-----------------+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
Ejercicio 2 — Triggers (processingTime)

In [7]:
# Detener streams previos activos
for q in spark.streams.active:
    q.stop()

# Ejercicio 2: Triggers - Intervalo fijo (processingTime = "10 seconds")
wordCountTrigger = (
    cantidadDF.writeStream
    .format("console")
    .outputMode("complete")
    .trigger(processingTime="10 seconds")
    .start()
)

wordCountTrigger.awaitTermination()

-------------------------------------------
Batch: 0
-------------------------------------------
+-------+-----+
|palabra|count|
+-------+-----+
+-------+-----+

-------------------------------------------
Batch: 1
-------------------------------------------
+---------+-----+
|  palabra|count|
+---------+-----+
|   prueba|    1|
|  trigger|    1|
|    batch|    1|
|    spark|    1|
|segundos,|    1|
|     diez|    1|
+---------+-----+



ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/valentin/Documents/manejo masivo de datos/spark-practicas/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 